# The OLS assumptions: practice notebook

**Causal and Predictive Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

All the Python from the module's web page, plus short exercises (cells marked `# YOUR CODE HERE`). Four small data sets:

| File | What it is |
|---|---|
| `hprices.csv` | 88 houses: `price` (in $1,000s) and size `sqrft` |
| `diamonds.csv` | 380 diamonds: `price`, `carats`, `color` |
| `wines.csv` | 230 wines: `Price`, rating `Score`, `Year` |
| `cps12.csv` | 3,955 full-time workers (Current Population Survey, 2012): hourly `earnings`, `male`, `female` |

Parts 1 and 2 use **simulated** data: when we generate the data ourselves, we know the true coefficients and can see whether OLS finds them.

**In Colab:** *File → Save a copy in Drive* first, so your changes are kept. Run cells with **Shift+Enter**.

## 0. Setup and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.options.display.float_format = "{:,.4f}".format
rng = np.random.default_rng(472)          # a seeded random number generator: everyone gets the same draws

In [ ]:
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/GBA472/A4/"   # the module's web address; filled in when the notebook is published, so the data load in Colab


def load(name):
    """Read a data file from the folder next to this notebook, or from the module's website."""
    for folder in [Path("data"), Path("web/data"), Path("../web/data")]:
        if (folder / name).exists():
            return pd.read_csv(folder / name)
    return pd.read_csv(SITE + "data/" + name)


housedata = load("hprices.csv")
diamonds = load("diamonds.csv")
winedata = load("wines.csv")
cps = load("cps12.csv")
print(housedata.shape, diamonds.shape, winedata.shape, cps.shape)

## 1. Why trust OLS? The three properties, by simulation

A population in which the assumptions hold: $\text{price}=11+0.14\times\text{size}+u$, with $u$ unrelated to size.
`one_sample(n)` draws $n$ houses and returns the OLS slope.

In [ ]:
def one_sample(n, skewed=False):
    size = rng.normal(2000, 580, n).clip(900, 4100)
    u = 60 * (rng.exponential(1, n) - 1) if skewed else rng.normal(0, 60, n)     # mean 0, SD 60, unrelated to size
    df = pd.DataFrame({"size": size, "price": 11 + 0.14 * size + u})
    return smf.ols("price ~ size", data=df).fit().params["size"]


slopes = np.array([one_sample(100) for _ in range(1000)])
print("true slope 0.1400;  average of 1,000 estimates:", round(slopes.mean(), 4), "  their SD:", round(slopes.std(), 4))
plt.hist(slopes, bins=40, color="#2563eb", edgecolor="white")
plt.axvline(0.14, color="black", ls="--")
plt.xlabel("slope estimate in a sample of 100 houses")
plt.show()

The estimates are centered on the truth (**unbiased**) and bell-shaped (**approximately normal**).

**Exercise 1.** Consistency: repeat with samples of 400 houses. By what factor does the SD of the estimates fall? (Recall $SE(\hat\beta_1)=SER/(\sqrt n\,s_X)$.)

In [ ]:
# YOUR CODE HERE


## 2. Assumption 1: wages, education, and ability

A simulated labor market where we know the truth: **one more year of education raises the wage by $2.00**.
Ability also raises wages, and people with more ability get more education.

In [ ]:
n = 5000
ability = rng.normal(0, 1, n)
educ = np.round(14 + 1.5 * ability + rng.normal(0, 2, n)).clip(8, 22)      # education rises with ability
wage = 5 + 2.0 * educ + 4.0 * ability + rng.normal(0, 6, n)                # the truth
sim = pd.DataFrame({"wage": wage, "educ": educ, "ability": ability})
print("correlation of education and ability:", round(sim.educ.corr(sim.ability), 2))

short = smf.ols("wage ~ educ", data=sim).fit()                 # ability is in the error term
print(short.summary().tables[1])

The coefficient on education is about 2.9, not 2.0: with ability in the error term, $E(u\mid X)\neq0$, and OLS gives education the credit for the ability that comes with it.

Two ways to restore Assumption 1: **put ability in the regression** (if you can measure it), or **assign education at random**.

In [ ]:
long = smf.ols("wage ~ educ + ability", data=sim).fit()        # ability is no longer in the error term
print("with ability in the regression:", round(long.params["educ"], 3))

# an experiment: shuffle education across people, so that it is unrelated to ability, and generate wages again
rct = sim.copy()
rct["educ"] = rng.permutation(rct.educ.values)
rct["wage"] = 5 + 2.0 * rct.educ + 4.0 * rct.ability + rng.normal(0, 6, n)
print("education assigned at random:  ", round(smf.ols("wage ~ educ", data=rct).fit().params["educ"], 3))

**Exercise 2.** More data does not fix the bias. Draw a sample of 200,000 workers the same way, regress wage on education, and look at the coefficient and its standard error.

In [ ]:
# YOUR CODE HERE


## 3. One extreme observation

The 88 houses, and the same data with one typo: a 3,500 square foot house entered at $50K instead of $500K.

In [ ]:
est = smf.ols("price ~ sqrft", data=housedata).fit()
typo = pd.concat([housedata, pd.DataFrame({"sqrft": [3500], "price": [50.0]})], ignore_index=True)
est_typo = smf.ols("price ~ sqrft", data=typo).fit()
print("88 houses:      slope", round(est.params["sqrft"], 4), "  R2", round(est.rsquared, 3))
print("with the typo:  slope", round(est_typo.params["sqrft"], 4), "  R2", round(est_typo.rsquared, 3))

plt.scatter(typo.sqrft, typo.price, s=14)
grid = np.array([1000, 4000])
plt.plot(grid, est.params["Intercept"] + est.params["sqrft"] * grid, "k--", label="88 houses")
plt.plot(grid, est_typo.params["Intercept"] + est_typo.params["sqrft"] * grid, color="#ea580c", label="with the typo")
plt.xlabel("size (square feet)"); plt.ylabel("price ($1,000s)"); plt.legend()
plt.show()

**Exercise 3.** Instead of the typo, add a 4,800 square foot house priced exactly on the regression line of the 88 houses. What happens to the slope and to its standard error? Why?

In [ ]:
# YOUR CODE HERE


## 4. Heteroskedasticity and robust standard errors

### 4.1 One more argument in `fit()`

In [ ]:
est = smf.ols("price ~ sqrft", data=housedata).fit()                      # default: homoskedasticity-only
est_r = smf.ols("price ~ sqrft", data=housedata).fit(cov_type="HC1")      # heteroskedasticity-robust
print(est.summary().tables[1])
print(est_r.summary().tables[1])

**Exercise 4.** Compute the 95% confidence interval for the slope by hand with each standard error ($\hat\beta_1\pm1.96\times SE$). Does zero lie in either interval?

In [ ]:
# YOUR CODE HERE


### 4.2 The same comparison on four data sets

In [ ]:
examples = [(housedata, "price ~ sqrft", "sqrft"), (diamonds, "price ~ carats", "carats"),
            (winedata, "Price ~ Score", "Score"), (cps, "earnings ~ male", "male")]
for df, formula, x in examples:
    a = smf.ols(formula, data=df).fit()
    b = smf.ols(formula, data=df).fit(cov_type="HC1")
    print(f"{formula:16s} coef {a.params[x]:10.4f}   default SE {a.bse[x]:9.4f}   robust SE {b.bse[x]:9.4f}"
          f"   t {a.tvalues[x]:6.2f}   z {b.tvalues[x]:6.2f}")

### 4.3 Looking for it: the residual plot

In [ ]:
housedata["resid"] = est.resid
plt.scatter(housedata.sqrft, housedata.resid, s=18)
plt.axhline(0, color="gray", lw=1)
plt.xlabel("size (square feet)"); plt.ylabel("residual ($1,000s)")
plt.show()

housedata["size"] = pd.qcut(housedata.sqrft, 3, labels=["small", "middle", "large"])
print(housedata.groupby("size", observed=True).resid.std().round(1))     # the spread of the residuals, by size

**Exercise 5.** Make the residual plot for the wine regression (`Price ~ Score`). Does the spread of the residuals change with the score? Is that in line with the two standard errors in 4.2?

In [ ]:
# YOUR CODE HERE


### 4.4 Which standard error is right? A simulation (optional)

Data with errors whose spread grows with $X$; true slope 2. The SD of the estimates across samples is what a standard error should match.

In [ ]:
def het_sample(n=100):
    x = 10 * rng.uniform(0, 1, n) ** 2                       # many small X's, a few large ones
    u = 3 * (0.15 + x / 10) / 0.568 * rng.normal(0, 1, n)    # the spread of the error grows with X
    df = pd.DataFrame({"x": x, "y": 5 + 2 * x + u})
    a = smf.ols("y ~ x", data=df).fit()
    b = smf.ols("y ~ x", data=df).fit(cov_type="HC1")
    return a.params["x"], a.bse["x"], b.bse["x"]


res = np.array([het_sample() for _ in range(1000)])
b, se, se_r = res.T
print("average estimate:", round(b.mean(), 3), "(truth 2)")
print("SD of the estimates:", round(b.std(), 3), "  average default SE:", round(se.mean(), 3), "  average robust SE:", round(se_r.mean(), 3))
print("95% intervals that contain the truth:  default", round(np.mean(abs(b - 2) <= 1.96 * se) * 100, 1), "%   robust",
      round(np.mean(abs(b - 2) <= 1.96 * se_r) * 100, 1), "%")

The estimates are centered on 2 (heteroskedasticity does not bias them); the default standard error is too small, so its "95%" intervals cover the truth less than 95% of the time; the robust standard error is about right.

---
*That's all. Next: regression lines that are not straight, and prediction with confidence and prediction intervals.*